In [ ]:
# SPDX-FileCopyrightText: 2024 Dan J. Bower <dbower@eaps.ethz.ch>
#
# SPDX-License-Identifier: GPL-3.0-or-later

import logging

import numpy as np

from atmodeller import (
    ChemicalSpecies,
    EquilibriumModel,
    IronWustiteBuffer,
    Parameters,
    Planet,
    ReservoirSpecies,
    debug_logger,
    earth,
    get_solubility_models,
)

logger = debug_logger()
logger.setLevel(logging.INFO)
# For more output use DEBUG instead of INFO
# logger.setLevel(logging.DEBUG)


# Basic usage

This notebook is available at `notebooks/tutorials/basic_usage.ipynb` and is easiest to obtain by downloading the source code.

We will walk through the process of defining species, setting up a model, running a basic equilibrium calculation, and accessing the output.

The scenario considered here is an Earth-sized planet with a molten silicate mantle in equilibrium with its atmosphere. Some atmospheric species have solubility relations that regulate how much of their inventory dissolves. The objective is to determine how the volatile budget partitions between the molten interior (magma ocean) and the atmosphere, and what the associated atmospheric speciation is.

## Create species

### Gas

Define the gas phase species to include in the interior-atmosphere model.

In [ ]:
# Gas species
H2O_g = ChemicalSpecies.create_gas("H2O")
H2_g = ChemicalSpecies.create_gas("H2")
O2_g = ChemicalSpecies.create_gas("O2")
CO_g = ChemicalSpecies.create_gas("CO")
CO2_g = ChemicalSpecies.create_gas("CO2")
CH4_g = ChemicalSpecies.create_gas("CH4")

gas_species = (H2O_g, H2_g, O2_g, CO_g, CO2_g, CH4_g)

### Dissolved

Create dissolved species, i.e. species that are dissolved in the molten interior according to solubility relations.

In [ ]:
# Various solubility models are available in Atmodeller and can be accessed via the
# `get_solubility_models` function
solubility_models = get_solubility_models()

# Set to True to include the mass of dissolved species in the phase (melt) mass. When this is set
# to False it is assumed that the mass of dissolved species is negligible compared to the mass of
# the melt, i.e. the dilute approximation is applied.
include_in_phase_mass = False

# Melt species
H2O_d: ReservoirSpecies = ReservoirSpecies.create_dissolved(
    "H2O",
    solubility=solubility_models["H2O_peridotite_sossi23"],
    include_in_phase_mass=include_in_phase_mass,
)
CO2_d: ReservoirSpecies = ReservoirSpecies.create_dissolved(
    "CO2",
    solubility=solubility_models["CO2_basalt_dixon95"],
    include_in_phase_mass=include_in_phase_mass,
)

melt_species = (H2O_d, CO2_d)

## Create model

Create a model by first defining a thermodynamic state, such as a molten Earth.

In [ ]:
planet = Planet.from_species(
    gas_species,
    melt_species=melt_species,
    temperature=2000,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=earth.radius,
)

Define system constraints, which can be a combination of activity (fugacity) constraints and mass constraints. For early planetary evolution, it is typical to impose an oxygen fugacity using the Iron-Wüstite buffer, regulated by Fe-FeO equilibrium. Importantly, a constraint must be prescribed for every element present in the system.

In [ ]:
# Fugacity constraint for fO2 is set by the Iron-Wuestite buffer
activity_constraints = {"O2_g": IronWustiteBuffer()}

# Mass constraints based on the mass of Earth's oceans and a C/H ratio of 1
oceans = 1
ch_ratio = 1
h_kg = earth.oceans_to_hydrogen_mass(oceans)
c_kg = ch_ratio * h_kg
mass_constraints = {"C": c_kg, "H": h_kg}

Create the model.

In [ ]:
parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

Solve the model.

In [ ]:
output = model.solve_with_default()

Check the model converged.

In [ ]:
output.solver_stats_to_logger(logger)

## Output

*Atmodeller* provides several ways to access the output. For use in your workflows it is likely most convenient to get a nested dictionary of output quantities.

In [ ]:
# By default the returned arrays are JAX arrays, but you can convert any arrays to numpy
# as follows, which is probably your preferred format for further analysis and plotting.
output_dict_default = output.to_dict(to_numpy=True)

print("Dictionary keys for 'named_arrays' output format:", list(output_dict_default.keys()))

# E.g., access the elemental masses in the gas phase
print("Gas phase elemental masses:", output_dict_default["gas"]["elements"]["mass"])

You can also obtain the output in a different format, for example by elements and species.

In [ ]:
output_dict_elements_species = output.to_dict(output_format="elements_species", to_numpy=False)

print(
    "Dictionary keys for 'elements_species' output format:",
    list(output_dict_elements_species.keys()),
)

# E.g., access quantities associated with hydrogen
print("Hydrogen quantities:", output_dict_elements_species["H"])

Another useful output option is a dictionary of dataframes, in which the same ``output_format`` options can be specified.

In [ ]:
output_format = "named_arrays"  # (default) or "elements_species"

dataframe_dict = output.to_dataframes(output_format=output_format)

The dataframes can be exported to a pickle file

In [ ]:
# Uncomment to export to a pickle file
# output.to_pickle("basic_usage.pkl", output_format=output_format)  # pyright: ignore

or to Excel.

In [ ]:
# output.to_excel("basic_usage.xlsx", output_format=output_format)  # pyright: ignore

## Batching

*Atmodeller* supports batching: you can provide arrays of input quantities and *Atmodeller* will automatically solve the system for each set of inputs using broadcasting rules. This enables efficient computation of multiple scenarios in a single call.

For example, let's define a new planet and batch over the temperature.

In [ ]:
temperature_batch = np.array([1500, 2000, 2500])

planet = Planet.from_species(
    gas_species,
    melt_species=melt_species,
    temperature=temperature_batch,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=earth.radius,
)

parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

output = model.solve_with_default()

output.solver_stats_to_logger(logger)

The output will similarly reflect the fact that temperature has been batched. You can also force all the arrays to match the batch size by setting ``expand_to_batch=True``.

In [ ]:
output_dict = output.to_dict(to_numpy=True, expand_to_batch=True)

# Let's check that we recover the imposed temperature batch
print("temperature:", output_dict["state"]["temperature"])

# Because we set expand_to_batch=True, even non-batched quantities are returned as arrays
print("radius:", output_dict["state"]["surface_radius"])

Finally, you can batch multiple parameters at the same time, as long as they have the same batch size. For example, you can batch both temperature and planetary radius, and they will be paired together for each calculation.

In [ ]:
temperature_batch = np.array([1500, 2000, 2500])
radius_batch = np.array([earth.radius, earth.radius * 1.5, earth.radius * 2])

planet = Planet.from_species(
    gas_species,
    melt_species=melt_species,
    temperature=temperature_batch,
    mantle_melt_fraction=1.0,
    planet_mass=earth.mass,
    core_mass_fraction=earth.core_mass_fraction,
    surface_radius=radius_batch,
)

parameters = Parameters(
    planet, activity_constraints=activity_constraints, mass_constraints=mass_constraints
)

model = EquilibriumModel(parameters)

output = model.solve_with_default()

output.solver_stats_to_logger(logger)

output_dict = output.to_dict(to_numpy=True, expand_to_batch=True)

# Let's check that we recover the imposed temperature batch
print("temperature:", output_dict["state"]["temperature"])

# Because we set expand_to_batch=True, even non-batched quantities are returned as arrays
print("radius:", output_dict["state"]["surface_radius"])